In [1]:
""" Imports """
%load_ext autoreload
%autoreload 2

import os
import matplotlib.pyplot as plt
import numpy as np

import torch
from sklearn.metrics import classification_report, accuracy_score, mean_squared_error

from simulation_encoder.models.model_retrieval import load_models
from simulation_encoder.loaders.loader_retrieval import load_loaders

In [2]:
""" Utility functions"""
def permutation_importance_aggregated_temporal(model, test_loader, baseline_score, criterion, num_iterations=1):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    model.to(device)
    model.eval()
    
    all_embeddings = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            embedding = model.encode(images)
            all_embeddings.append(embedding)
            all_labels.append(labels)
    
    all_embeddings = torch.cat(all_embeddings, dim=0)
    all_labels = torch.cat(all_labels, dim=0).long()
    
    num_features = all_embeddings.shape[1]
    importance_scores = torch.zeros(num_features)
    
    with torch.no_grad():    
        for _ in range(num_iterations):
            iteration_importance = []
        
            for feature_idx in range(num_features):
                permuted_embeddings = all_embeddings.clone()
            
                permuted_feature = permuted_embeddings[:, feature_idx].clone()
                perm_idx = torch.randperm(permuted_embeddings.shape[0])
                permuted_embeddings[:, feature_idx] = permuted_feature[perm_idx]
            
                permuted_logits = model.decode_timepoint(permuted_embeddings)
            
                permuted_score = criterion(permuted_logits, all_labels)
                importance = permuted_score - baseline_score
                iteration_importance.append(importance)
        
            importance_scores += torch.Tensor(iteration_importance)

        importance_scores /= num_iterations
        feature_importance = sorted(enumerate(importance_scores), key=lambda x: x[1], reverse=True)

    return feature_importance

def visualize_feature_tuning_temporal(model, latent_samples, feature_idx, variation_range=(-3, 3), steps=7):
    half_steps = steps // 2
    feature_values = torch.linspace(variation_range[0], variation_range[1], steps)

    fig, axes = plt.subplots(1, steps, figsize=(3 * steps, 3))
    
    with torch.no_grad():
        original_image = model.decode_image(latent_samples).squeeze().detach().cpu().numpy()
        
        for col, val in enumerate(feature_values):
            modified_latent = latent_samples.clone().squeeze()
            
            if col == half_steps:
                image_to_plot = original_image
                title = "Original"
                cmap = 'gray'
                vmin = vmax = None
            else:
                modified_latent[feature_idx] += val
                modified_image = model.decode_image(modified_latent.unsqueeze(0)).squeeze().detach().cpu().numpy()
                
                diff_image = modified_image - original_image
                image_to_plot = diff_image
                title = f"{val.item():+.2f}"
                cmap = 'RdBu'
                vmax = np.max(np.abs(diff_image))
                vmin = -vmax
            
            axes[col].imshow(image_to_plot, cmap=cmap, vmin=vmin, vmax=vmax)
            axes[col].set_title(title)
            axes[col].axis('off')
    
    plt.tight_layout()
    plt.show()

def permutation_importance_aggregated_spatial(model, test_loader, baseline_score, criterion, num_iterations=1):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    model.to(device)
    model.eval()

    all_embeddings = []
    all_images = []
    
    with torch.no_grad():
        for images, _ in test_loader:
            images_cpu = images.cpu()
            embedding = model.encode(images.to(device)).to("cpu")
            all_embeddings.append(embedding)
            all_images.append(images_cpu)

    all_embeddings = torch.cat(all_embeddings, dim=0)
    all_images = torch.cat(all_images, dim=0).squeeze()

    num_features = all_embeddings.shape[1]
    importance_scores = torch.zeros(num_features, dtype=torch.float16)
    
    with torch.no_grad():
        for _ in range(num_iterations):
            iteration_importance = []
            
            for feature_idx in range(num_features):
                permuted_embeddings = all_embeddings.clone()
                
                permuted_feature = permuted_embeddings[:, feature_idx].clone()
                perm_idx = torch.randperm(permuted_embeddings.shape[0])
                permuted_embeddings[:, feature_idx] = permuted_feature[perm_idx]

                permuted_preds = model.decode_image(permuted_embeddings.to(device)).squeeze()
                
                permuted_score = criterion(permuted_preds, all_images.to(device)).item()
                importance = permuted_score - baseline_score
                iteration_importance.append(importance)
            
            importance_scores += torch.Tensor(iteration_importance)

        importance_scores /= num_iterations
        feature_importance = sorted(enumerate(importance_scores), key=lambda x: x[1], reverse=True)

    model.cpu()
    return feature_importance

def visualize_feature_tuning_spatial(model, latent_sample, feature_idx, variation_range=(-3, 3), steps=7):    
    half_steps = steps // 2
    feature_values = torch.linspace(variation_range[0], variation_range[1], steps)

    fig, axes = plt.subplots(1, steps, figsize=(3 * steps, 3))
    
    with torch.no_grad():
        original_image = model.decode_image(latent_sample).squeeze().detach().cpu().numpy()
        
        for col, val in enumerate(feature_values):
            modified_latent = latent_sample.clone().squeeze()
            
            if col == half_steps:
                image_to_plot = original_image
                title = "Original"
                cmap = 'gray'
                vmin = vmax = None
            else:
                modified_latent[feature_idx] += val
                modified_image = model.decode_image(modified_latent.unsqueeze(0)).squeeze().detach().cpu().numpy()

                diff_image = modified_image - original_image
                image_to_plot = diff_image
                title = f"{val.item():+.2f}"
                cmap = 'RdBu'
                vmax = np.max(np.abs(diff_image))
                vmin = -vmax
            
            axes[col].imshow(image_to_plot, cmap=cmap, vmin=vmin, vmax=vmax)
            axes[col].set_title(title)
            axes[col].axis('off')
    
    plt.tight_layout()
    return fig

In [3]:
study_name = "arch_simple_gastruloid_processed_16_simple"
image_dir = "data"
data_type = "gastruloid"


results_dir = f"results/{study_name}"

all_models = load_models(results_dir, best_models_flag=True)
all_loaders = load_loaders(results_dir, image_dir)

Loading model ae_simple


RuntimeError: Error(s) in loading state_dict for AE:
	size mismatch for decoder_timepoint.6.weight: copying a param with shape torch.Size([9, 32]) from checkpoint, the shape in current model is torch.Size([15, 32]).
	size mismatch for decoder_timepoint.6.bias: copying a param with shape torch.Size([9]) from checkpoint, the shape in current model is torch.Size([15]).

# Timepoint

In [ ]:
num_iterations = 1
top_temporal_features = {}
log_loss = torch.nn.CrossEntropyLoss(reduction='mean')
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

for model_name, dataset in all_models.items():
    top_temporal_features[model_name] = {}
    for dataset_name, model in dataset.items():
        test_loader = all_loaders[dataset_name].get_dataloader("test")
        y_true_labels, y_pred_logits = [], []

        model.to(device)
        model.eval()
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                labels = labels.to(device)
                embedding = model.encode(images)
                logits = model.decode_timepoint(embedding)
                # Append entire batch tensors rather than extending:
                y_true_labels.append(labels)
                y_pred_logits.append(logits)

        # Use torch.cat to combine the batch tensors and keep them on the same device.
        y_true = torch.cat(y_true_labels, dim=0).long()
        y_logits = torch.cat(y_pred_logits, dim=0)
        baseline_acc = log_loss(y_logits, y_true)
        print(f"\nBaseline accuracy for {model_name} | {dataset_name}: {baseline_acc:.4f}")

        feature_importances = permutation_importance_aggregated_temporal(
            model, test_loader, baseline_acc, log_loss, num_iterations
        )
        top_temporal_features[model_name][dataset_name] = feature_importances
        print(f"Averaged feature importances (over {num_iterations} iterations):")
        for idx, imp in feature_importances:
            print(f"\tLatent dimension {idx}: importance = {imp:.4f}")

In [ ]:
for model_name, dataset in all_models.items():
    for dataset_name, model in dataset.items():
        most_important_features = top_temporal_features[model_name][dataset_name][0][0]
        model.to(device)
        model.eval()
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                labels = labels.to(device)
                embeddings = model.encode(images)

                visualize_feature_tuning_temporal(
                    model, embeddings, most_important_features, variation_range=(-3, 3), steps=7
                )
                break
                

# Spatial

In [ ]:
num_iterations = 1
top_spatial_features = {}
mse_loss = torch.nn.MSELoss(reduction='mean')
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

for model_name, dataset in all_models.items():
    top_spatial_features[model_name] = {}
    for dataset_name, model in dataset.items():
        test_loader = all_loaders[dataset_name].get_dataloader("test")
        y_true_list, y_pred_list = [], []

        model.to(device)
        model.eval()
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)

                embedding = model.encode(images)
                image_pred = model.decode_image(embedding)

                y_true_list.append(images)
                y_pred_list.append(image_pred)

        y_true = torch.cat(y_true_list, dim=0)
        y_pred = torch.cat(y_pred_list, dim=0)
        baseline_acc = mse_loss(y_pred, y_true).item()
        print(f"\nBaseline accuracy for {model_name} | {dataset_name}: {baseline_acc:.4f}")

        feature_importances = permutation_importance_aggregated_spatial(
            model, test_loader, baseline_acc, mse_loss, num_iterations
        )
        top_spatial_features[model_name][dataset_name] = feature_importances
        print(f"Averaged feature importances (over {num_iterations} iterations):")
        for idx, imp in feature_importances:
            print(f"\tLatent dimension {idx}: importance = {imp:.4f}")
    
        model.cpu()

In [ ]:
for model_name, dataset in all_models.items():
    for dataset_name, model in dataset.items():
        most_important_features = top_temporal_features[model_name][dataset_name][0][0]
        model.to(device)
        model.eval()
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                labels = labels.to(device)
                embeddings = model.encode(images)

                visualize_feature_tuning_spatial(
                    model, embeddings, most_important_features, variation_range=(-3, 3), steps=7
                )
                break